## ODS competition autumn 2025

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier, GradientBoostingClassifier, RandomForestClassifier

from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('train_final.csv', index_col=0)

In [3]:
df.head()

,user_id,ts,gate_id
0,18,2022-07-29 09:08:54,7
1,18,2022-07-29 09:09:54,9
2,18,2022-07-29 09:09:54,9
3,18,2022-07-29 09:10:06,5
4,18,2022-07-29 09:10:08,5


In [4]:
df_final = pd.read_csv('test_final.csv', index_col=0)
df_final.head()

,ts,gate_id,user_word
37518,2023-01-03 08:21:00,9,gini
37519,2023-01-03 08:21:00,9,gini
37520,2023-01-03 08:21:18,5,gini
37521,2023-01-03 08:21:19,5,gini
37522,2023-01-03 08:21:39,10,gini


In [ ]:
# Feature Engineering 
df['ts'] = pd.to_datetime(df['ts'])
df = df.sort_values('ts')

df['hour'] = df['ts'].dt.hour
df['min'] = df['ts'].dt.minute
df['day_of_week'] = df['ts'].dt.dayofweek
df['day_of_month'] = df['ts'].dt.day

# Cyclical Features 
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Gate + DOW
df['gate_dow'] = df['gate_id'].astype(str) + '_' + df['day_of_week'].astype(str)

# Lag Feature 
df['time_diff'] = df['ts'].diff().dt.total_seconds().fillna(0)

df = df.drop(columns=['ts'])

df['hour_minute'] = df['hour'] * 60 + df['min'] # Признак для частей дня

bins = [0, 6, 12, 18, 23]
labels = ['night', 'morning', 'day', 'evening']
df['part_of_day'] = pd.cut(df['hour'], bins=bins, labels=labels, include_lowest=True)

df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)  # 5 и 6 - суббота и воскресенье

df = df.drop(columns=['hour', 'day_of_week', 'day_of_month'])

le = LabelEncoder()
df['user_id_encoded'] = le.fit_transform(df['user_id'])
Y = df['user_id_encoded']

X = df.drop(columns=['user_id', 'user_id_encoded'])


### Preprocessing

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# ColumnTransformer setup
numerical_features = ['time_diff', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'min', 'hour_minute']
categorical_features = ['gate_id', 'gate_dow', 'part_of_day', 'is_weekend']

# preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        # 1. Standard Scaler для числовых признаков (для регуляризации)
        ('num', StandardScaler(), numerical_features),
        # 2. OneHotEncoder для категориальных признаков
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_features)
    ],
    remainder='passthrough' # Оставить остальные признаки как есть (на данный момент их нет)
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)

In [12]:
model=RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced', max_depth=20, min_samples_split=5, min_samples_leaf=2)
model.fit(X_train_processed, Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,20
,min_samples_split,5
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
# Предсказание на валидационном наборе
Y_preds = model.predict(X_val_processed)
acc = accuracy_score(Y_val, Y_preds)

# Декодирование меток обратно в исходные ID пользователей
Y_val_original = le.inverse_transform(Y_val)
Y_preds_original = le.inverse_transform(Y_preds)

print(f"\nValidation Accuracy (Model): {acc:.4f}")

In [ ]:
df_letters = pd.read_csv('test_final.csv')

letter_ids = df_letters['user_word'].copy()

# Apply feature engineering
df_letters['ts'] = pd.to_datetime(df_letters['ts'])
df_letters = df_letters.sort_values('ts')

df_letters['hour'] = df_letters['ts'].dt.hour
df_letters['min'] = df_letters['ts'].dt.minute
df_letters['day_of_week'] = df_letters['ts'].dt.dayofweek
df_letters['day_of_month'] = df_letters['ts'].dt.day

df_letters['hour_sin'] = np.sin(2 * np.pi * df_letters['hour'] / 24)
df_letters['hour_cos'] = np.cos(2 * np.pi * df_letters['hour'] / 24)
df_letters['dow_sin'] = np.sin(2 * np.pi * df_letters['day_of_week'] / 7)
df_letters['dow_cos'] = np.cos(2 * np.pi * df_letters['day_of_week'] / 7)

df_letters['gate_dow'] = df_letters['gate_id'].astype(str) + '_' + df_letters['day_of_week'].astype(str)
df_letters['time_diff'] = df_letters['ts'].diff().dt.total_seconds().fillna(0)
df_letters = df_letters.drop(columns=['ts'])

X_letters = df_letters.drop(columns=['user_word'])

X_letters_processed = preprocessor.transform(X_letters)
predicted_encoded = model.predict(X_letters_processed)

# decode predictions to original ids
predicted_ids = le.inverse_transform(predicted_encoded)

mapping_df = pd.DataFrame({
    'user_word': letter_ids,
    'preds': predicted_ids
})

In [ ]:
prob_df = mapping_df.groupby(['letter_id', 'predicted_id']).size().reset_index(name='count')
prob_df['probability'] = prob_df['count'] / prob_df.groupby('letter_id')['count'].transform('sum')

prob_df = prob_df.sort_values(['letter_id', 'probability'], ascending=[True, False])

final_mapping_dict = {}
used_predicted_ids = set()

for letter_id in prob_df['letter_id'].unique():
    available = prob_df[
        (prob_df['letter_id'] == letter_id) & 
        (~prob_df['predicted_id'].isin(used_predicted_ids))
    ]
    
    if not available.empty:
        # max probability
        selected = available.iloc[0]
        final_mapping_dict[letter_id] = selected['predicted_id']
        used_predicted_ids.add(selected['predicted_id'])
    else:
        final_mapping_dict[letter_id] = -999

# make DataFrame
final_map = pd.DataFrame(list(final_mapping_dict.items()), 
                             columns=['user_word', 'preds'])

final_map

,user_word,preds
0,aucroc,49
1,binary,12
2,blue,6
3,categorical,14
4,coefficient,15
5,collinear,33
6,distributed,11
7,epsilon,1
8,f1,18
9,fit,3


In [17]:
final_map.to_csv('answer_final.csv', index=False)

### Rewrite latter

In [ ]:
# Reusable Feature Engineering Function 
def apply_features(df):
    df = df.copy()
    
    # Ensure 'ts' is the timestamp column
    ts_col_name = [c for c in df.columns if 'ts' in c.lower() or 'time' in c.lower()][0]
    df['ts'] = pd.to_datetime(df[ts_col_name])
    df = df.sort_values('ts')

    df['hour'] = df['ts'].dt.hour
    df['min'] = df['ts'].dt.minute
    df['day_of_week'] = df['ts'].dt.dayofweek
    df['day_of_month'] = df['ts'].dt.day

    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    df['gate_dow'] = df['gate_id'].astype(str) + '_' + df['day_of_week'].astype(str)

    df['time_diff'] = df['ts'].diff().dt.total_seconds().fillna(0)
    
    df = df.drop(columns=['ts'])
    
    return df